In [17]:
import pandas as pd
import json

In [18]:
df_terms_final = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

In [19]:
print("Terms shape:", df_terms_final.shape)
print("Products shape:", df_products.shape)
df_products.head(3)

Terms shape: (20394, 5)
Products shape: (20394, 18)


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.usagetype,attributes.operation,attributes.eksproducttype,attributes.instancetype,attributes.regionCode,attributes.servicename,attributes.tiertype,attributes.capabilitytype,attributes.ekscapabilityunits,attributes.tenancy,attributes.memorytype,attributes.cputype,attributes.storagetype
0,A53K5SWT3G3CRHF5,Compute,AmazonEKS,Asia Pacific (Malaysia),AWS Region,APS7-EKS-Auto:r8i.8xlarge-management-hours,EKSAutoUsage,AutoMode,r8i.8xlarge,ap-southeast-5,Amazon Elastic Container Service for Kubernetes,None,None,None,None,None,None,None
1,TCP2YWGAHGPW67N9,Compute,AmazonEKS,South America (Sao Paulo),AWS Region,SAE1-EKS-Auto:m6g.metal-management-hours,EKSAutoUsage,AutoMode,m6g.metal,sa-east-1,Amazon Elastic Container Service for Kubernetes,None,None,None,None,None,None,None
2,MCK76ZZJZMWS7HDE,Compute,AmazonEKS,Middle East (Bahrain),AWS Region,MES1-EKS-Auto:r5d.xlarge-management-hours,EKSAutoUsage,AutoMode,r5d.xlarge,me-south-1,Amazon Elastic Container Service for Kubernetes,None,None,None,None,None,None,None


In [20]:
if 'attributes.servicename' in df_products.columns:
    df_products.drop(columns=['attributes.servicename'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (20394, 17)


In [21]:
df_products.drop(columns=['attributes.regionCode'], errors='ignore', inplace=True)


In [22]:
df_master_eks = pd.merge(
    df_products, 
    df_terms_final, 
    on='sku', 
    how='inner'
).reset_index(drop=True)

In [23]:
df_master_eks.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.usagetype,attributes.operation,attributes.eksproducttype,attributes.instancetype,attributes.tiertype,attributes.capabilitytype,attributes.ekscapabilityunits,attributes.tenancy,attributes.memorytype,attributes.cputype,attributes.storagetype,rateCode,description,unit,priceUSD
0,A53K5SWT3G3CRHF5,Compute,AmazonEKS,Asia Pacific (Malaysia),AWS Region,APS7-EKS-Auto:r8i.8xlarge-management-hours,EKSAutoUsage,AutoMode,r8i.8xlarge,None,None,None,None,None,None,None,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),hours,0.27348
1,TCP2YWGAHGPW67N9,Compute,AmazonEKS,South America (Sao Paulo),AWS Region,SAE1-EKS-Auto:m6g.metal-management-hours,EKSAutoUsage,AutoMode,m6g.metal,None,None,None,None,None,None,None,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),hours,0.47002
2,MCK76ZZJZMWS7HDE,Compute,AmazonEKS,Middle East (Bahrain),AWS Region,MES1-EKS-Auto:r5d.xlarge-management-hours,EKSAutoUsage,AutoMode,r5d.xlarge,None,None,None,None,None,None,None,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),hours,0.04224
3,FQKX5BN95S5F5T2E,Compute,AmazonEKS,Asia Pacific (Sydney),AWS Region,APS2-EKS-Auto:m5.4xlarge-management-hours,EKSAutoUsage,AutoMode,m5.4xlarge,None,None,None,None,None,None,None,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),hours,0.11520
4,USZRSB47YGRTASDR,Compute,AmazonEKS,Asia Pacific (Malaysia),AWS Region,APS7-EKS-Auto:i8ge.large-management-hours,EKSAutoUsage,AutoMode,i8ge.large,None,None,None,None,None,None,None,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),hours,0.03074


In [24]:
summary_data = []
for col in df_products.columns:
    sample_vals = list(df_products[col].dropna().unique()[:5])
    summary_data.append({'Column': col, 'Sample_Values': sample_vals})

df_summary = pd.DataFrame(summary_data)

# Εμφάνιση χωρίς περικοπές
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_summary

,Column,Sample_Values
0,sku,"[A53K5SWT3G3CRHF5, TCP2YWGAHGPW67N9, MCK76ZZJZMWS7HDE, FQKX5BN95S5F5T2E, USZRSB47YGRTASDR]"
1,productFamily,[Compute]
2,attributes.servicecode,[AmazonEKS]
3,attributes.location,"[Asia Pacific (Malaysia), South America (Sao Paulo), Middle East (Bahrain), Asia Pacific (Sydney), US West (Oregon)]"
4,attributes.locationType,"[AWS Region, AWS Outposts, AWS Local Zone]"
5,attributes.usagetype,"[APS7-EKS-Auto:r8i.8xlarge-management-hours, SAE1-EKS-Auto:m6g.metal-management-hours, MES1-EKS-Auto:r5d.xlarge-management-hours, APS2-EKS-Auto:m5.4xlarge-management-hours, APS7-EKS-Auto:i8ge.large-management-hours]"
6,attributes.operation,"[EKSAutoUsage, CreateOperation, ArgoCDUsage, ACKUsage, ProvisionedControlPlaneUsage]"
7,attributes.eksproducttype,"[AutoMode, HybridNodes]"
8,attributes.instancetype,"[r8i.8xlarge, m6g.metal, r5d.xlarge, m5.4xlarge, i8ge.large]"
9,attributes.tiertype,"[HAStandard, HAProvisioned2XL, HAProvisioned4XL, HAProvisioned8XL, HAExtended]"
